## Reading from bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")
df.display()

## Trimming & normalize the data

In [0]:
query = """
    SELECT 
        cst_id,
        cst_key,
        TRIM(cst_firstname) AS cst_firstname,
        TRIM(cst_lastname) AS cst_lastname,
        CASE UPPER(TRIM(cst_marital_status)) 
            WHEN 'M' THEN 'Married'
            WHEN 'S' THEN 'Single' 
            ELSE 'n/a'
        END AS cst_marital_status,
        CASE UPPER(TRIM(cst_gndr)) 
            WHEN 'M' THEN 'Male'
            WHEN 'F' THEN 'Female'
            ELSE 'n/a'
        END AS cst_gndr,
        cst_create_date
    FROM workspace.bronze.crm_cust_info
"""
df = spark.sql(query)
df.display()

## Update table schema

In [0]:
rename_schema = {
    'cst_id': 'customer_id',
    'cst_key': 'customer_key',
    'cst_firstname': 'firstname',
    'cst_lastname': 'lastname',
    'cst_marital_status': 'marital_status',
    'cst_gndr': 'gender',
    'cst_create_date': 'created_at'
}
for old_name, new_name in rename_schema.items():
   df = df.withColumnRenamed(old_name, new_name)

### Load transform data into silver layer

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_cust_info")

In [0]:
display(spark.table("workspace.silver.crm_cust_info"))